# 02 December Benchmark Detector

This notebook runs the stable December 2000–2018 benchmark workflow using the reusable repo modules. It opens cloud ERA5, computes the JPCZ polygon-mean divergence series, applies the Shinoda-style threshold, and saves first-pass benchmark outputs.

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

from jpcz_catalog.config import (
    BASELINE_PEAK_DATE_UTC,
    DECEMBER_BENCHMARK_YEARS,
    JPCZ_POLYGON_VERTICES,
)
from jpcz_catalog.detect import (
    compute_polygon_mean_divergence_series,
    count_events_from_threshold,
    prepare_detection_geometry,
    threshold_from_std,
    threshold_sensitivity,
)
from jpcz_catalog.era5 import open_arco_era5, open_baseline_window, open_december_month
from jpcz_catalog.plotting import plot_polygon_mean_timeseries
from jpcz_catalog.verification import (
    render_december_benchmark_summary,
    render_first_pass_summary,
    render_threshold_sensitivity_summary,
    write_text_summary,
)

ds = open_arco_era5()
print("valid_time_start:", ds.attrs.get("valid_time_start"))
print("valid_time_stop:", ds.attrs.get("valid_time_stop"))
print("valid_time_stop_era5t:", ds.attrs.get("valid_time_stop_era5t"))

baseline_ds = open_baseline_window(ds)
geometry = prepare_detection_geometry(
    baseline_ds.longitude,
    baseline_ds.latitude,
    JPCZ_POLYGON_VERTICES,
)
baseline_hourly, baseline_D, _ = compute_polygon_mean_divergence_series(
    baseline_ds,
    geometry=geometry,
)
baseline_peak_time = baseline_D.dropna("time").idxmin("time").values
plot_polygon_mean_timeseries(
    baseline_hourly,
    baseline_D,
    peak_time=baseline_peak_time,
    title=f"Baseline Event Check (expected peak on {BASELINE_PEAK_DATE_UTC})",
)


In [ ]:
import pandas as pd
import xarray as xr

yearly_checkpoint_dir = Path(DRIVE_OUTPUT_DIR) / "december_yearly" if PERSIST_OUTPUTS_TO_DRIVE else Path("outputs/verification/december_yearly")
yearly_checkpoint_dir.mkdir(parents=True, exist_ok=True)
print("Yearly checkpoint dir:", yearly_checkpoint_dir)

total_hourly_points = 0
dec_D_series = []

for year in DECEMBER_BENCHMARK_YEARS:
    yearly_d_path = yearly_checkpoint_dir / f"december_{year}_D.nc"

    if yearly_d_path.exists():
        D_valid = xr.open_dataarray(yearly_d_path).load()
        hourly_points = int(D_valid.sizes['time'] + 11)
        total_hourly_points += hourly_points
        dec_D_series.append(D_valid)
        print(
            f"{year}: loaded checkpoint hourly={hourly_points}, "
            f"valid_D={D_valid.sizes['time']}, "
            f"peak_D={float(D_valid.min().values):.3e} s^-1"
        )
        continue

    dec_ds = open_december_month(ds, year)
    hourly_dec, D_dec, _ = compute_polygon_mean_divergence_series(
        dec_ds,
        geometry=geometry,
    )

    D_valid = D_dec.dropna("time")
    D_valid.to_netcdf(yearly_d_path)
    total_hourly_points += int(hourly_dec.sizes['time'])
    dec_D_series.append(D_valid)

    print(
        f"{year}: hourly={hourly_dec.sizes['time']}, "
        f"valid_D={D_valid.sizes['time']}, "
        f"peak_D={float(D_valid.min().values):.3e} s^-1"
    )

all_dec_D = xr.concat(dec_D_series, dim="time").sortby("time")

D_mean, D_std, D_threshold = threshold_from_std(all_dec_D, n_std=2.0)
events_df = count_events_from_threshold(all_dec_D, D_threshold)
sensitivity_df = threshold_sensitivity(all_dec_D, n_std_values=(2.0, 2.1, 2.2, 2.3, 2.4, 2.5))

events_out = Path("outputs/verification/december_events_first_pass.csv")
events_df.to_csv(events_out, index=False)
D_out = Path("outputs/verification/december_D_timeseries.nc")
all_dec_D.to_netcdf(D_out)
sensitivity_out = Path("outputs/verification/december_threshold_sensitivity.csv")
sensitivity_df.to_csv(sensitivity_out, index=False)

benchmark_summary = render_december_benchmark_summary(
    total_events=len(events_df),
    D_mean=D_mean,
    D_std=D_std,
    threshold=D_threshold,
)
benchmark_summary_path = write_text_summary("outputs/verification/december_benchmark_summary.md", benchmark_summary)

sensitivity_counts = {
    row["threshold_label"]: int(row["event_count"])
    for _, row in sensitivity_df.iterrows()
}
threshold_summary_path = write_text_summary(
    "outputs/verification/december_threshold_sensitivity.md",
    render_threshold_sensitivity_summary(sensitivity_counts),
)
first_pass_summary_path = write_text_summary(
    "outputs/verification/final_first_pass_summary.md",
    render_first_pass_summary(),
)

if PERSIST_OUTPUTS_TO_DRIVE:
    for path in [
        events_out,
        D_out,
        sensitivity_out,
        benchmark_summary_path,
        threshold_summary_path,
        first_pass_summary_path,
    ]:
        shutil.copy2(path, Path(DRIVE_OUTPUT_DIR) / path.name)

    print("Copied outputs to:", DRIVE_OUTPUT_DIR)

print("Total hourly December points:", total_hourly_points)
print("Total valid December D points:", all_dec_D.sizes["time"])
print("Detected major December events:", len(events_df))
print("Saved D series to:", D_out)
print("Saved threshold sensitivity table to:", sensitivity_out)
print(benchmark_summary)
sensitivity_df
